# Fuda · handwritten kana recognizer (Colab)

Trains a small CNN on the **ETL Character Database** to recognize the 92 basic kana
(46 hiragana + 46 katakana) and exports it to Core ML for the Fuda Mac app.

**Before you start** (in Google Drive, folder `MyDrive/fuda-hw/`):
- `code/` — the files from `Tools/handwriting/` in the FudaMac repo (`kana.py`, `etl.py`, `preprocess.py`, `prepare.py`, `train.py`, `eval_folder.py`)
- `etl/` — the ETL zip files you downloaded from https://etlcdb.db.aist.go.jp/ : `ETL1.zip` (katakana) and `ETL8G.zip` (hiragana). `ETL6.zip` / `ETL7.zip` are optional extras.
- `my_samples/` *(optional)* — your own drawings exported from Fuda → 書 Handwriting → Collect

Runtime → Change runtime type → **GPU** (A100 is plenty; this model trains in a few minutes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/fuda-hw'
!ls -la $ROOT $ROOT/etl

In [ ]:
!pip -q install coremltools onnx onnxscript
!apt-get -qq install -y fonts-noto-cjk > /dev/null   # Japanese labels in the charts
!rm -rf /root/.cache/matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy the code and unzip the ETL files to the fast local disk.
!rm -rf /content/hw && mkdir -p /content/hw/etl && cp $ROOT/code/*.py /content/hw/
!cd /content/hw/etl && for z in $ROOT/etl/*.zip; do unzip -q -o "$z"; done
!find /content/hw/etl -maxdepth 2 | head -50

## 1 · Build the dataset
Reads every ETL file, keeps the 92 kana, cleans each image (crop → square + margin → 64×64) and
**splits by writer** (80/10/10): nobody's handwriting appears in both training and test.
Check `grid.png`: one random sample per class — any wrong label is obvious here.

In [ ]:
%cd /content/hw
!python prepare.py --etl etl --out kana.npz --grid grid.png
from IPython.display import Image, display
display(Image('grid.png'))

## 2 · Train
~1.6M-parameter ResNet-style CNN, GPU augmentation (rotation, scale, shear, shift, stroke thickness),
class-balanced sampling (ETL1 has ~9× more katakana than ETL8G has hiragana), label smoothing,
OneCycle LR. The **test set is scored once**, with the checkpoint that was best on validation.

In [ ]:
!python train.py --data kana.npz --out run1 --epochs 40 --bs 512

In [ ]:
import json
m = json.load(open('run1/metrics.json'))
print('test accuracy:', m['test_acc'], ' hiragana:', m['hiragana_test_acc'], ' katakana:', m['katakana_test_acc'])
print('test images / writers:', m['test_images'], m['test_writers'])
print('worst classes:', m['worst_classes'])
print('top confusions:', [(c['true'], c['predicted'], c['count']) for c in m['top_confusions'][:12]])
display(Image('run1/confusion.png')); display(Image('run1/errors.png'))

## 3 · Your own handwriting (the honest number)
Draw samples in Fuda (書 Handwriting → Collect, ~10 per kana), copy the exported folder to
`MyDrive/fuda-hw/my_samples/`, then:

In [ ]:
!python eval_folder.py --model run1/best.pt --folder $ROOT/my_samples

## 4 · Save the model for the app
Copies `KanaRecognizer.mlpackage` (Core ML, for Fuda), `kana.onnx` (for a web demo) and the
metrics/charts to Drive. Put `KanaRecognizer.mlpackage` in `FudaMac/Resources/` and rebuild Fuda.

In [ ]:
!mkdir -p $ROOT/results && cp -r run1/KanaRecognizer.mlpackage run1/kana.onnx run1/metrics.json run1/*.png grid.png $ROOT/results/
!cd $ROOT/results && zip -qr KanaRecognizer.mlpackage.zip KanaRecognizer.mlpackage && ls -la